# Simulating Stochastic Wildfire Spread and Vegetation Recovery with the WildfireGenerator Landlab Component

*Authors: M. W. de Almeida and B. Campforts*

---


- **Stochastic fire ignition** driven by fuel availability and aridity
- **Fire spread** across a raster grid using a logistic slope-modulated probability
- **Post-fire vegetation recovery** as an asymptotic function of aridity

The component follows the fire-climate-vegetation theory of [Pausas & Paula (2012)](https://doi.org/10.1111/j.1466-8238.2011.00753.x) and [Pausas & Ribeiro (2013)](https://doi.org/10.1111/geb.12043), and is part of the **FireLands 1.0** landscape evolution model (de Almeida et al., 2026).

---

## Contents

1. [Setup](#1-setup)
2. [Grid and component initialisation](#2-grid-and-component-initialisation)
3. [Live fire spread animation (100 years)](#3-live-fire-spread-animation)
4. [Post-run analysis](#4-post-run-analysis)

## 1. Setup

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np

from landlab import RasterModelGrid
from landlab.components import FlowAccumulator, WildfireGenerator

## 2. Grid and component initialisation

We create a 50×50 raster grid with:
- A gently tilted topography with random noise to create a realistic drainage network
- A `FlowAccumulator` to compute drainage area, which the component uses to identify river firebreaks
- A random initial fuel load between 0.3 and 0.8

Key parameters controlling fire behaviour:

| Parameter | Value | Effect |
|---|---|---|
| `potential_fires` | 20 | Mean ignition attempts per year |
| `aridity` | 0.7 | Dry climate → larger, more severe fires |
| `minimum_river_threshold` | 1e6 m² | Only large rivers act as firebreaks |
| `upslope_preference` | 0.8 | Strong upslope spread bias |

In [ ]:
np.random.seed(42)
nrows, ncols = 50, 50
mg = RasterModelGrid((nrows, ncols), xy_spacing=100.0)

# Tilted topography with noise
z = mg.add_zeros("topographic__elevation", at="node")
z[:] = mg.node_y * 0.01 + mg.node_x * 0.005 + np.random.rand(mg.number_of_nodes) * 0.5

# Flow accumulation for river firebreaks
fa = FlowAccumulator(mg, flow_director="D8")
fa.run_one_step()

# Initial fuel load
fuel = mg.add_zeros("fuel_availability", at="node")
fuel[:] = 0.3 + np.random.rand(mg.number_of_nodes) * 0.5

# Instantiate WildfireGenerator
wg = WildfireGenerator(
    mg,
    potential_fires=20,
    aridity=0.7,
    minimum_river_threshold=1e6,
    upslope_preference=0.8,
    severity_exponent=0.64,
    seed=42,
)

print(f"Grid: {nrows}x{ncols} nodes, {mg.dx} m spacing")
print(f"Initial mean fuel: {fuel.mean():.2f}")

## 3. Live fire spread animation

The two panels update every year:

- **Left** — fuel availability across the grid. Green = high fuel, yellow = low fuel. Depletion from fire and recovery between events are both visible.
- **Right** — fire severity for the **current year only** (reset each timestep). White = no fire this year, dark red = high severity fire.

Adjust `plt.pause()` to control animation speed.

In [ ]:
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

# Reset component for a fresh run
np.random.seed(42)
fuel[:] = 0.3 + np.random.rand(mg.number_of_nodes) * 0.5
wg = WildfireGenerator(
    mg,
    potential_fires=20,
    aridity=0.7,
    minimum_river_threshold=1e6,
    upslope_preference=0.8,
    severity_exponent=0.64,
    seed=42,
)

# Pre-run all timesteps and store snapshots
n_steps = 100
dt = 1.0
fuel_snapshots = []
severity_snapshots = []

severity_map = np.zeros(mg.number_of_nodes)

for step in range(n_steps):
    wg.run_one_step(dt)

    severity_map[:] = 0.0
    for record in wg.last_step_fires:
        severity_map[record["burned_nodes"]] = record["severity_factor"]

    fuel_snapshots.append(fuel.copy().reshape(mg.shape))
    severity_snapshots.append(severity_map.copy().reshape(mg.shape))

# Build animation from snapshots
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

norm_fuel = mcolors.Normalize(vmin=0, vmax=1)
norm_sev = mcolors.Normalize(vmin=0, vmax=1)

im_fuel = axes[0].imshow(fuel_snapshots[0], origin="lower", cmap="YlGn", norm=norm_fuel)
axes[0].set_title("Fuel availability")
axes[0].set_xlabel("x (nodes)")
axes[0].set_ylabel("y (nodes)")
fig.colorbar(im_fuel, ax=axes[0], label="Fuel [-]", shrink=0.8)

im_sev = axes[1].imshow(
    severity_snapshots[0], origin="lower", cmap="Reds", norm=norm_sev
)
axes[1].set_title("Fire severity (current year only)")
axes[1].set_xlabel("x (nodes)")
axes[1].set_ylabel("y (nodes)")
fig.colorbar(im_sev, ax=axes[1], label="Severity [-]", shrink=0.8)

title = fig.suptitle("Year 1", fontsize=13)
plt.tight_layout()


def update(frame):
    im_fuel.set_data(fuel_snapshots[frame])
    im_sev.set_data(severity_snapshots[frame])
    n_fires = sum(1 for r in wg.fire_log if r["year"] == frame + 1)
    title.set_text(f"Year {frame + 1}  |  Fires this year: {n_fires}")
    return im_fuel, im_sev, title


anim = FuncAnimation(fig, update, frames=n_steps, interval=150, blit=False)
plt.close(fig)  # prevent static figure from also showing

HTML(anim.to_jshtml())

## 4. Post-run analysis

After the 100-year run, we inspect the full fire log to examine:
- The distribution of fire sizes
- How severity varies over time
- Cumulative burned area and mean fuel through time

In [ ]:
log = wg.fire_log
sizes = np.array([r["fire_size (km2)"] for r in log])
severities = np.array([r["severity_factor"] for r in log])

print(f"Total fire events : {len(log)}")
print(f"Mean fire size    : {sizes.mean():.4f} km²")
print(f"Max fire size     : {sizes.max():.4f} km²")
print(f"Mean severity     : {severities.mean():.2f}")
print(f"\nFire log (first 5 records):")

In [ ]:
log = wg.fire_log
sizes = np.array([r["fire_size (km2)"] for r in log])
severities = np.array([r["severity_factor"] for r in log])
years = np.array([r["year"] for r in log])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Fire size distribution
axes[0].hist(sizes, bins=20, color="firebrick", alpha=0.8, edgecolor="white")
axes[0].set_xlabel("Fire size (km²)")
axes[0].set_ylabel("Count")
axes[0].set_title("Fire size distribution")

# Severity over time
sc = axes[1].scatter(
    years,
    severities,
    c=sizes,
    cmap="OrRd",
    alpha=0.8,
    edgecolors="grey",
    linewidths=0.4,
)
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Severity factor")
axes[1].set_title("Severity over time (colour = fire size)")
fig.colorbar(sc, ax=axes[1], label="Fire size (km²)")

# Number of fires per year
unique_years, counts = np.unique(years, return_counts=True)
axes[2].bar(unique_years, counts, color="steelblue", alpha=0.8)
axes[2].set_xlabel("Year")
axes[2].set_ylabel("Number of fires")
axes[2].set_title("Fire count per year")

plt.tight_layout()
plt.show()

In [ ]:
# Cumulative fire area and mean fuel through time
log = wg.fire_log
sizes = np.array([r["fire_size (km2)"] for r in log])
years = np.array([r["year"] for r in log])
pct_veg = np.array([r["pct of vegetation removed"] for r in log])

unique_years = np.arange(1, n_steps + 1)
cumulative_area = np.cumsum([sizes[years == y].sum() for y in unique_years])
mean_veg_per_year = np.array(
    [pct_veg[years == y].mean() if np.any(years == y) else 0.0 for y in unique_years]
)

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(unique_years, cumulative_area, color="firebrick", lw=2)
ax1.set_xlabel("Year")
ax1.set_ylabel("Cumulative burned area (km²)", color="firebrick")
ax1.tick_params(axis="y", labelcolor="firebrick")

ax2 = ax1.twinx()
ax2.plot(unique_years, mean_veg_per_year, color="forestgreen", lw=2, linestyle="--")
ax2.set_ylabel("Mean % vegetation removed", color="forestgreen")
ax2.tick_params(axis="y", labelcolor="forestgreen")

fig.suptitle("Cumulative fire impact over 100 years", fontsize=13)
plt.tight_layout()
plt.show()